 # <center>Practice Notebook - Lesson 7c - plotting in `pandas` </center>

This notebook will work through some basics of plotting with matplotlib, seaborn, and pandas.

References for the material in this notebook include 
* Chapter 9 of your _Python for Data Analysis_ (2nd edition)
* <a href="https://matplotlib.org/3.3.2/contents.html#">matplotlib</a>, <a href="https://seaborn.pydata.org/">seaborn</a>, and 
* <a href="https://pandas.pydata.org/pandas-docs/stable/user_guide/visualization.html">pandas</a>.

and others as posted on Canvas

---

### Matplotlib and numpy, without pandas

First, we will import matplotlib and numpy.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

Let's create a figure where plots of some data will live; then we create two subplots side by side.

In [ ]:
rand_fig = plt.figure()
plot_1 = rand_fig.add_subplot(1, 2, 1)
plot_2 = rand_fig.add_subplot(1, 2, 2)

Then, we'll plot some randomly-generated data:

In [ ]:
plot_1.plot(np.random.binomial(1000, 0.42,20), ':', drawstyle='steps-post')
plot_1.plot(np.random.binomial(1000, 0.42,20), 'k--')
plot_2.scatter(range(20),np.random.randn(20), color='r', label='random integers')
plot_2.scatter(range(20),range(20), color='b', label='linear')
plot_2.legend(loc='best')
rand_fig

Note that if you re-run the code above, additional data will be added to the plot. In the plots above, there are assigned line colors, line styles, and labels. Can you tell where each is indicated in the code? There are many formatting options that can be explored! You will likely want to set labels for the axes and title of your plot. To demonstrate, we graph the position of an object that at each time step can move 1 unit in either direction, or can stay put.

In [ ]:
fig_ex_2 = plt.figure()
ex_2 = fig_ex_2.add_subplot(1, 1, 1)
ex_2.plot(np.random.randint(-1,2,10).cumsum())
ex_2.set_title('Random Walk')
ex_2.set_xlabel('Time')
ex_2.set_ylabel('Position')
ex_2.set_xticks(np.arange(0,10,2))

We can also do things such as annotating a graph and drawing shapes - see Chapter 9 of your text for further examples. Finally, let's save our random walk figure as a png:

In [ ]:
fig_ex_2.savefig('random_walk.png')

---

### Plotting with seaborn & pandas

This section was inspired by the October 28, 2020 New York Times story [How Does Your State Make Electricity?](https://www.nytimes.com/interactive/2020/10/28/climate/how-electricity-generation-changed-in-your-state-election.html?unlocked_article_code=3JqmTapNSWCSITGiGejBp8OOySeHTtmjPY2OBn0O8E92wIT5U3Bb5RwYEuf9DTMTR0X6eC7bL1-KS2Yj3fQaFqrrlXFfSMCrUDk61nij9tUV3CTZV520gmfFP311CNTolcGuanSxT3LMqx49mMq-pv3tWtHSOReKuW1ZReuzmLfzmga3W864--_STzaBgSXWfRQR93P04sF4vZSy_lPUQpzQkOxrgs0_EM-fETxTF2sTyLjcgngiEsvyrGF0wWMOaZ6MTt4yxZdh0GD1MPB-pMSNbtKvR2FSx89BH3ZGnI2RXW6yt4if8HlOyRv7t3CBbzeQlyVORyLkvcZKXERll_2LYyxGkeJnxMtiP__Wh9R0Jqd3USojzq47id7fAMrHz44OxbKZBf7thRQKnPdltInI5vU&smid=share-url).  The data was downloaded from [The US Energy Information Administration](https://www.eia.gov/electricity/data/state/), specifically from [this link](https://www.eia.gov/electricity/data/state/annual_generation_state.xls) in an Excel notebook containing data for all states, and translated to the data just for Washington.  We've provided a file for you on the BGSC with just Washington, skipping a few steps.  

🎯 Copy the file `energy-gen-WA.csv` from `ds150_course_files/data/lesson7/` into your working directory for this lesson.

Let's import pandas, and read the file `energy-gen-WA.csv` into memory.

In [ ]:
import pandas as pd
energy_WA = pd.read_csv('energy-gen-WA.csv')
energy_WA.head(20)

🧠 Take a moment to use what you learned in the lesson on pandas to explore this data.

One thing you will want to check before we try to plot column values using seaborn is that the columns you think are numerical are actually numerical:

In [ ]:
energy_WA.dtypes

Oh no! We want the number of megawatt-hours (MWhs) generated by each source to be numerical values, but right now they are objects. No problem: let's remove the commas in the values and convert to integers.

In [ ]:
energy_WA['GENERATION_MWh'] = energy_WA['GENERATION_MWh'].str.replace(",","").astype(int)
energy_WA.dtypes

Now, we'll graph the energy generation against the year.

In [ ]:
import seaborn as sns
sns.lineplot(data=energy_WA, x='YEAR', y='GENERATION_MWh')

Notice that there are multiple rows corresponding to the same year: seaborn interpreted this by graphing the mean across all rows of the same year, along with a confidence interval (shaded). But this probably isn't what we wanted. Instead, we can separate out the rows with the same year according to the energy source, as follows. 

In [ ]:
sns.lineplot(data=energy_WA, x='YEAR', y='GENERATION_MWh', hue='ENERGY_SOURCE')

Now, of course, our issue is a fairly unreadable graph due to a giant legend. If we want, we can take away the legend.

In [ ]:
sns.lineplot(data=energy_WA, x='YEAR', y='GENERATION_MWh', hue='ENERGY_SOURCE', legend=False)

However, we actually do want to know which energy source is generating each line. So, we can resize the figure to make room for the legend as follows:

In [ ]:
fig, ax = plt.subplots(figsize=(20,10))
sns.lineplot(data=energy_WA, ax=ax, x='YEAR', y='GENERATION_MWh', hue='ENERGY_SOURCE')

If we only care about particular energy sources, we may want to make each energy source into its own column using a pivot table. Note that we fill all empty entries with a "0".

In [ ]:
pvt_energy_WA = pd.pivot_table(energy_WA, index=['YEAR'],columns=['ENERGY_SOURCE']).fillna(0)
pvt_energy_WA.head()

Now we can plot and adjust the location of the legend, the title, and the axes labels.

In [ ]:
sns.lineplot(data=pvt_energy_WA['GENERATION_MWh','Coal'], label = 'Coal')
sns.lineplot(data=pvt_energy_WA['GENERATION_MWh','Hydroelectric Conventional'], label = 'Hydroelectric')
sns.lineplot(data=pvt_energy_WA['GENERATION_MWh','Nuclear'], label = 'Nuclear')
plt.legend(loc='lower left')
plt.title('Washington Energy Production')
plt.xlabel('Year')
plt.ylabel('MWh')

We can also use pandas to plot. Suppose we want to create a stacked bar plot that gives the breakdown of megawatt-hours produced for each energy source in a given year. First, we remove the `'Total'` column, as this is just the sum of the other columns, and will appear as the total height of our stacked columns.

In [ ]:
pvt_energy_WA = pvt_energy_WA.drop(columns = [('GENERATION_MWh', 'Total')])

Finally, we'll create a stacked bar plot, using `alpha` to specify the opacity of the bars.

In [ ]:
fig, ax = plt.subplots(figsize=(20,10))
pvt_energy_WA.plot.bar(stacked=True, alpha=0.7, ax=ax)
plt.legend(loc='lower left', )
plt.title('Washington Energy Production')
plt.xlabel('Year')
plt.ylabel('MWh')

---

## When you're done

🧠 After completing the guided practice above, log onto Canvas to complete Quiz 7c. You may wish to keep this notebook open while you work.